# MateConv mini inference

In [1]:
import itertools
import re
import json
import jsonlines
import psutil
import ujson
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from datasets import load_dataset
import os
from tqdm import tqdm
import torch
from model.model import Transformer  # 确保路径正确
from model.LMConfig import LMConfig   # 导入 LMConfig

/root/miniconda3/envs/MateConv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 定义BOS和EOS标记
bos_token = "<s>"
eos_token = "</s>"

In [3]:
# 加载训练好的分词器路径
tokenizer = AutoTokenizer.from_pretrained('/root/autodl-tmp/MateConv/model/mateconv_tokenizer', use_fast=False)
print(f'加载的tokenizer词表大小: {len(tokenizer)}')

加载的tokenizer词表大小: 6400


In [4]:
# 创建配置对象
lm_config = LMConfig()

In [5]:
# 初始化 Transformer 模型
model = Transformer(lm_config)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
device

device(type='cuda')

In [8]:
model.to(device)

# 检查模型结构和参数
print(model)

Transformer(
  (tok_embeddings): Embedding(6400, 512)
  (dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlock(
      (attention): Attention(
        (wq): Linear(in_features=512, out_features=512, bias=False)
        (wk): Linear(in_features=512, out_features=256, bias=False)
        (wv): Linear(in_features=512, out_features=256, bias=False)
        (wo): Linear(in_features=512, out_features=512, bias=False)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
      (feed_forward): FeedForward(
        (w1): Linear(in_features=512, out_features=1408, bias=False)
        (w2): Linear(in_features=1408, out_features=512, bias=False)
        (w3): Linear(in_features=512, out_features=1408, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=512, 

In [9]:
# 加载模型权重【这里要修改为你的模型地址】
model.load_state_dict(torch.load('out/pretrain_512.pth', map_location=device))
model.eval()  # 切换到评估模式

Transformer(
  (tok_embeddings): Embedding(6400, 512)
  (dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlock(
      (attention): Attention(
        (wq): Linear(in_features=512, out_features=512, bias=False)
        (wk): Linear(in_features=512, out_features=256, bias=False)
        (wv): Linear(in_features=512, out_features=256, bias=False)
        (wo): Linear(in_features=512, out_features=512, bias=False)
        (attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
      (feed_forward): FeedForward(
        (w1): Linear(in_features=512, out_features=1408, bias=False)
        (w2): Linear(in_features=1408, out_features=512, bias=False)
        (w3): Linear(in_features=512, out_features=1408, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=512, 

In [43]:
# 准备输入文本
input_text = "决策树是机器学习中的一种算法，决策树是"
input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

# 生成文本
max_new_tokens = 500  # 生成 100 个 token
eos_token_id = tokenizer.eos_token_id  # 终止 token

# 在model.py中我们定义了generate方法、专用于生成
# 调用 model.generate() 进行生成
output_ids = next(model.generate(
    idx=input_ids,
    eos=eos_token_id,
    max_new_tokens=max_new_tokens,
    temperature=0.2,  # 控制创造性
    top_k=10,  # 限制 top-k 采样
    rp=1.2,  # 避免重复
    stream=False  # 关闭流式返回
))

# 解码生成的 token
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 打印最终生成的文本
print(generated_text)

基于人类数据的计算方法。
在本书中，作者通过对比分析了大脑和大脑之间的关系、思维方式以及行为模式等问题，并结合大量的数据资料进行了深入剖析，为读者提供了一个很好的参考工具。


In [45]:
# 准备输入文本
input_text = "我与你"
input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

# 生成文本
max_new_tokens = 500  # 生成 100 个 token
eos_token_id = tokenizer.eos_token_id  # 终止 token

# 在model.py中我们定义了generate方法、专用于生成
# 调用 model.generate() 进行生成
output_ids = next(model.generate(
    idx=input_ids,
    eos=eos_token_id,
    max_new_tokens=max_new_tokens,
    temperature=1,  # 控制创造性
    top_k=10,  # 限制 top-k 采样
    rp=1.2,  # 避免重复
    stream=False  # 关闭流式返回
))

# 解码生成的 token
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 打印最终生成的文本
print(generated_text)

在一起。
我爱你，也许你会觉得有点不公平了，但我会努力奋斗一辈子！我爱你，也许只是一段美好的时光而已。”他如此的坚定信念：“对不起，我的爱，是我的力量。希望能帮助你的人生，让生命更加美丽！”
“我们是最好的男人！”，他说到最后。“如果有一天你不会再见过我，我一定会珍惜这份温暖和关怀。
